# Model Development — Explanation
### A companion lecture to `Model Development.ipynb`

**Student:** Mathonsi Mphikeleli Mbongiseni (28574249) · UNISA MCom Quantitative Management

This notebook explains, from first principles, the dissertation's actual methodology: *Machine Learning-Augmented GARCH Models for Improving Value-at-Risk Estimation in Equity Portfolios*. It assumes no prior background in econometrics or deep learning and defines every concept before using it. Every number quoted below is the real value the technical notebook produced when it was executed, not an illustrative placeholder.

## The question this notebook is actually trying to answer

The dissertation's central claim is specific and testable: **a classical GARCH-family model, left to itself, does not capture everything predictable about tomorrow's volatility — and a machine learning layer, given the GARCH model's own output as its raw material rather than competing with it from scratch, can refine that forecast into a better one.** Everything in this notebook exists to test that claim honestly: three GARCH-family models are estimated properly and in their own right, a single LSTM (Long Short-Term Memory) network is layered on top of them as an adaptive correction step, and the resulting **Hybrid ML-GARCH** forecast is held to the same rigorous statistical backtesting as every classical model it is trying to improve on. The dissertation's architecture is a strict pipeline — return data, a GARCH layer, conditional variance forecasts, an LSTM refinement layer, a hybrid VaR forecast, and backtesting — and this notebook follows that pipeline in exactly that order.

## 1–2. Setup and Data Preparation

The setup cell imports four distinct toolkits, each doing a clearly separated job: pandas and numpy for data handling, matplotlib and seaborn for the figures, `arch` for the three GARCH-family models that form the classical layer of the architecture, and `keras` (running on a PyTorch computational backend, selected with `os.environ['KERAS_BACKEND'] = 'torch'` before Keras itself is imported) for the LSTM refinement layer. `scikit-learn` appears only for two small utility pieces — a feature scaler and a handful of accuracy metrics — never for a competing forecasting model of its own.

### Why a portfolio, and why now

Every notebook up to this point analysed the ten assets individually. This notebook makes a deliberate change of scope: it builds a single **equally-weighted portfolio** out of the eight individual equities, and treats the two market indices as diagnostic series rather than portfolio constituents. This is not a simplification for its own sake — **Value-at-Risk, in essentially every real-world application, is a portfolio-level concept.** No institution holds a single stock in isolation; it holds a book of positions, and what a risk committee actually needs is the risk of the whole book together, accounting for the fact that the EDA notebook already showed these assets do not move entirely independently of one another.

### Verifying the inputs, again

Before building anything on top of the cleaned data, the notebook re-runs a short battery of consistency checks against the files produced by the Data Cleaning notebook — matching row counts, zero missing values, no duplicate dates, and an explicit numerical check that recomputing returns directly from the cleaned prices reproduces the saved return file to within $10^{-8}$. This mirrors the validation habit introduced in the Data Cleaning notebook: **never assume a file produced by an earlier notebook is still correct — verify it again, immediately before it is used.** The notebook's own `assert` statement stops execution outright if any check fails, rather than silently building a dissertation's worth of models on data quietly known to be wrong.

**Reading the portfolio's own statistics.** The constructed portfolio shows an annualised return of **16.73%** and an annualised volatility of **22.58%**, with a skewness of **-0.47** and an excess kurtosis of **11.48** — confirming that combining eight assets into a portfolio reduces some idiosyncratic risk through diversification, but does **not** eliminate the fat-tailed, non-Normal behaviour documented for every individual asset in the EDA notebook. This single fact is the reason every model in this notebook is deliberately fitted with Student-t, rather than Normal, innovations.

## 3. Train/Test Split and Walk-Forward Design

### Why a financial time series cannot simply be shuffled

In many everyday machine learning problems, the standard practice is to randomly shuffle a dataset before splitting it into a training and a test set, on the assumption that each observation is independent of the others. **Neither assumption holds for a financial time series.** Observations are ordered in time for a reason, and nearby observations are not independent — the EDA notebook's own autocorrelation analysis showed exactly that. Shuffling before splitting would let information from the future leak backward into a model trained to predict the past, a mistake known as **look-ahead bias**, and it can make a model look deceptively accurate purely because it was secretly allowed to see ahead of itself.

The correct alternative, used throughout this notebook, is a **strictly chronological split**: the first 80% of the sample, in date order, is used for estimation (2010-01-05 to 2022-10-13, 3,217 observations); the final 20% (2022-10-14 to 2025-12-30, 805 observations) is held back entirely as an out-of-sample test period that no model is ever fitted directly on.

### Walk-forward re-estimation — and why the GARCH layer and the LSTM layer follow different clocks

Training every model once on the in-sample block and then generating predictions for the entire test period would understate how a model would actually be used in practice — a real risk desk periodically re-estimates its models as new data arrives rather than freezing them forever. The three GARCH-family models are therefore re-estimated on an **expanding window** (growing larger at every step, rather than a fixed-size rolling window that would also discard old data) every 63 trading days, roughly one calendar quarter, throughout the test period — cheap enough, as a closed-form maximum-likelihood fit, to repeat this often.

The LSTM refinement layer is different: it is trained **once**, on the in-sample block only, and is then applied walk-forward across the entire test period, consuming the GARCH layer's own quarterly-updated forecasts as its input without itself being retrained. This asymmetry is a deliberate design choice, not an inconsistency: retraining a neural network — a comparatively slow, iterative optimisation process — on the same quarterly cadence as a fast, closed-form econometric model would add considerable computational cost for little expected benefit, and is not standard practice in the hybrid GARCH-LSTM literature this dissertation builds on (e.g. Kim & Won, 2018, who train their LSTM once and evaluate it walk-forward in exactly this way).

## 4. GARCH-Family Volatility Models

### The general idea: today's shock and today's variance predict tomorrow's variance

Every model in this section rests on one central empirical fact, established directly from this project's own data in Section 7 of the EDA notebook: while tomorrow's *return* is close to unpredictable from past returns (near-zero autocorrelation), tomorrow's *variance* is not — the autocorrelation of squared returns is strong and persistent, the signature of **volatility clustering**. **GARCH**, standing for **Generalised AutoRegressive Conditional Heteroskedasticity**, is a formal mathematical description of exactly this pattern, introduced by Tim Bollerslev in 1986 building on Robert Engle's earlier ARCH model (work for which Engle shared the 2003 Nobel Memorial Prize in Economic Sciences), and it has remained the standard, industry-wide benchmark for volatility forecasting ever since.

### GARCH(1,1)

The baseline specification is

$$\sigma_{t}^2 = \omega + \alpha\, \varepsilon_{t-1}^2 + \beta\, \sigma_{t-1}^2$$

where $\varepsilon_{t-1}^2$ is yesterday's squared shock (return) and $\sigma_{t-1}^2$ is the model's own previous variance estimate. $\alpha$ measures how strongly a fresh shock raises variance; $\beta$ measures how strongly elevated variance persists once it is there; and their sum, $\alpha + \beta$, measures how quickly a spike in volatility decays back toward its long-run average — a sum close to 1 (here, $0.1277 + 0.8560 = 0.9837$) implies volatility shocks are extremely persistent, taking a long time to fade, exactly the slow-moving, regime-like behaviour visible in the EDA notebook's rolling-volatility charts. `dist='t'` fits a **Student-t** distribution for the shocks rather than a Normal one, with the degrees-of-freedom parameter ($\nu \approx 5.97$ here) estimated jointly by maximum likelihood — a low value, implying distinctly fatter tails than Normal, directly consistent with the excess kurtosis documented for every asset in the EDA notebook.

### GJR-GARCH(1,1) — allowing bad news to matter more

**GJR-GARCH**, named for Glosten, Jagannathan and Runkle (1993), extends the baseline with one additional term:

$$\sigma_t^2 = \omega + \alpha\, \varepsilon_{t-1}^2 + \gamma\, \varepsilon_{t-1}^2 I_{t-1} + \beta\, \sigma_{t-1}^2, \qquad I_{t-1} = 1 \text{ if } \varepsilon_{t-1} < 0$$

so a negative shock adds an *extra* $\gamma \varepsilon_{t-1}^2$ on top of the usual $\alpha \varepsilon_{t-1}^2$ term, while a positive shock of the same size does not. This asymmetry, known as the **leverage effect**, is a well-documented pattern in equity markets dating back to Fischer Black's work in 1976 — one common explanation is that a falling share price mechanically raises a company's financial leverage (the weight of its now-fixed debt against its shrunken equity value), making the remaining equity riskier. The fitted $\gamma = 0.1873$ is positive and, alongside a much smaller symmetric $\alpha = 0.0136$, confirms this asymmetry empirically in this portfolio's own history: almost all of the model's reaction to a shock comes from the *asymmetric* term, meaning it is overwhelmingly *negative* shocks that this model has learned to treat as informative about future risk.

### EGARCH(1,1) — modelling the logarithm of variance directly

**EGARCH** (Exponential GARCH, Nelson, 1991) takes a different mathematical route to the same underlying idea: rather than modelling $\sigma_t^2$ directly and needing to constrain every parameter to keep that quantity positive, it models $\log(\sigma_t^2)$, which can take any value at all without breaking the model — variance itself, recovered by exponentiating, is then automatically guaranteed to be positive with no constraints required. Its own asymmetry term, $\gamma = -0.1362$, plays the same conceptual role as GJR-GARCH's $\gamma$: a negative sign means negative shocks raise conditional variance more than positive shocks of the same size, the same leverage effect finding reached by GJR-GARCH's structurally different route. EGARCH's $\beta = 0.9695$, the highest persistence of the three models, implies the smoothest, slowest-moving volatility forecast of the three — visible directly in the walk-forward comparison chart, where EGARCH's line is noticeably less jagged than GARCH's or GJR-GARCH's.

### Reading the walk-forward violation rates

Once re-estimated walk-forward across the 805-day test period, all three models produce broadly sensible violation rates: GARCH breaches its 95% VaR on 3.11% of days and its 99% VaR on 0.75% (both somewhat conservative relative to their 5.0% and 1.0% targets); EGARCH and GJR-GARCH both sit at 3.98% and around 1.1–1.2% respectively. All three are in a defensible range at first glance. Whether "defensible at first glance" survives formal statistical testing — specifically, whether these violations are *independent* through time rather than clustering together — is exactly the question Section 6 answers properly, and the answer turns out to matter a great deal.

## 5. LSTM Refinement Layer

This is the dissertation's central methodological contribution, so it is worth building up from genuine first principles rather than treating "LSTM" as a black box that simply appears.

### From a plain neural network to a recurrent one

An ordinary neural network (a Multi-Layer Perceptron) takes a fixed-size input, passes it through one or more layers of artificial neurons, and produces an output — but it has no built-in notion of *order* or *sequence*. Feeding it "Monday, Tuesday, Wednesday" produces the same internal computation as feeding it "Wednesday, Monday, Tuesday", because each input is processed independently of the others. That is a serious mismatch for a problem like this one, where the *order* in which volatility evolved over the past month is itself part of the signal — a month that ends with volatility rising is a genuinely different situation from one that ends with volatility falling, even if the two months contain the exact same set of daily values in different order.

A **Recurrent Neural Network (RNN)** fixes this by processing a sequence one step at a time while carrying a running internal summary — a **hidden state** — forward from each step to the next, so that by the time it reaches the last day in a sequence, that hidden state reflects everything the network has seen so far, in the order it was seen. In principle this is exactly the structure needed here. In practice, plain RNNs suffer badly from the **vanishing gradient problem**: the learning algorithm (backpropagation, extended across time as **backpropagation through time**) adjusts the network's weights based on how much each weight contributed to the final error, and for a plain RNN that contribution shrinks multiplicatively at every step going backward through a long sequence, until information from early in the sequence barely influences learning at all — a plain RNN effectively "forgets" anything more than a few steps in the past.

### Why LSTM specifically

**Long Short-Term Memory (LSTM)**, introduced by Hochreiter and Schmidhuber in 1997, was designed specifically to solve this problem. Alongside the ordinary hidden state, an LSTM cell maintains a second, separate internal pathway called the **cell state**, which acts as a kind of conveyor belt for information running through the sequence with only minor, carefully controlled modifications at each step — a structure that lets gradients flow backward through many time steps without vanishing the way they do in a plain RNN. What controls those modifications are three learned **gates**, each itself a small neural network layer that outputs a number between 0 and 1 at every time step, acting as a "how much" dial:

- the **forget gate** decides how much of the existing cell state to keep versus discard;
- the **input gate** decides how much of the current time step's new information to add to the   cell state;
- the **output gate** decides how much of the (possibly updated) cell state to expose as the   hidden state passed on to the next step, and ultimately to whatever comes after the LSTM layer.

All of these gates, and the transformations feeding them, are learned automatically from data during training — nothing about which patterns to "remember" or "forget" is hand-specified. This is precisely why an LSTM, rather than a plain RNN or a simple feed-forward network, is the right tool for a *refinement* layer here: it can learn, from the data itself, how much weight to give a volatility spike from three weeks ago versus one from yesterday, rather than being told a fixed rule in advance the way GARCH's own fixed $\alpha$ and $\beta$ parameters effectively are.

### 5.1 Feature construction — what the LSTM is actually shown

Critically, and by explicit design, **the LSTM never sees raw prices, and never independently re-derives volatility from scratch.** Its input at every day $t$ is a 21-trading-day sequence (roughly one calendar month) of seven values, all knowable by the close of day $t$: the portfolio's own return, the conditional volatility already produced by each of the three GARCH-family models, and the standardised residual implied by each ($r_t / \sigma_t$ — how many "model standard deviations" surprising that day's return was, according to each GARCH variant). This is what makes the architecture a genuine **refinement layer** rather than a competing, independent forecaster: every one of the six GARCH-derived inputs already encodes a GARCH-family model's own opinion about risk, and the LSTM's only job is to learn how to combine and correct those three opinions using patterns across time that no single GARCH equation, built around a fixed one-step recursion, can represent on its own. The prediction target is unchanged from every other model in this notebook: $\log(r_{t+1}^2 + \varepsilon)$, the same next-day log-variance a GARCH model is itself built to forecast, so the comparison in Section 6 is a genuinely fair, apples-to-apples one.

One implementation detail matters for correctness: for the in-sample period, each GARCH-family model's own *fitted* (in-sample) conditional volatility and standardised residuals are used directly, while for the out-of-sample test period, the same walk-forward forecasts already produced in Section 4.2 are used instead — never information a live deployment would not yet have had. Building 3,770 usable sequences of length 21 produces **3,197 training sequences** and, after the very last day (whose target does not exist, since there is no day after it to compute a next-day return from) is dropped, **804 test sequences**.

### 5.2 Feature scaling

The seven inputs sit on genuinely different scales — a daily return and a conditional volatility are both small decimals typically under 0.05, while a standardised residual is, by construction, of order 1 — and a neural network's gradient-based training converges far more reliably once every input is put on a comparable scale. A `StandardScaler`, identical in principle to the one introduced for the (now-removed) regularised linear models in earlier drafts of this dissertation's pipeline, is fitted **only on the training sequences** and applied unchanged to the test sequences, so no information about the test period's own scale leaks backward into training.

### 5.3 Architecture and training

The network is deliberately small: a single LSTM layer of 32 units reads each 21-day sequence and condenses it into a 32-number summary; a **dropout** layer then randomly deactivates 20% of that summary during every training step — a standard regularisation technique that prevents the network from becoming overly reliant on any single internal pathway, forcing it to learn more robust, redundant representations — before two ordinary dense layers map the result down to a single log-variance prediction. In total the network has **5,665 trainable parameters**, small by modern deep learning standards, deliberately so: with only 3,197 training sequences available, a much larger network would be at serious risk of overfitting.

Training minimises **mean squared error** using the **Adam optimiser**, a standard, adaptive variant of gradient descent, holding back the chronologically *final* 15% of the training sequences as a validation set (never shuffled into the earlier training portion, preserving the same chronological discipline as every split in this notebook) purely to monitor for overfitting during training. **Early stopping** halts training automatically once validation loss stops improving for 10 consecutive epochs, and restores the network's weights from whichever epoch achieved the *best* validation loss rather than simply keeping the final epoch's weights — training on this run stopped after **48 epochs**, with a final training loss of 5.42 and validation loss of 4.48, close enough to each other to suggest the network has not badly overfit its own training data.

### 5.4 From a log-variance prediction to a usable hybrid volatility forecast

The LSTM predicts $\log(\text{variance})$, exactly as every model in this dissertation's earlier development targeted, and turning that back into an ordinary variance requires undoing the logarithm with `exp(...)`. Doing so naively **understates** the true expected variance, because of **Jensen's inequality**: the average of the exponentials of a set of prediction errors is always somewhat larger than the exponential of their average, whenever those errors have any real spread at all, since the exponential function curves upward. The **Duan (1983) smearing correction** fixes this directly and simply: the model's own in-sample residuals (actual minus predicted log-variance, on the training data it was just fitted to) are exponentiated and averaged once, producing a single correction factor — **3.7706** on this run — and every out-of-sample forecast is multiplied by that factor after being exponentiated. Skipping this step is not a minor omission: without it, this kind of model's forecast volatility comes out roughly half of its true level, and the VaR built on top of it would be dangerously, silently too aggressive. With the correction applied, the Hybrid model's mean forecast volatility across the test period, **1.265%**, sits close to the actual realised test-period volatility of **1.206%** — a first, reassuring sign the correction has worked as intended, well before Section 6's formal backtests confirm it more rigorously.

## 6. Model Evaluation

All four volatility forecasts — GARCH, EGARCH, GJR-GARCH and the Hybrid ML-GARCH refinement — are compared on one identical evaluation window (804 days; the LSTM's sequence construction drops the very last test day, since its target does not exist, so every model is evaluated on that same slightly shorter window rather than mixing windows of different lengths).

### 6.1 Volatility forecast accuracy

Six metrics are computed for each model against a realised volatility proxy, $|r_{t+1}|$: **RMSE** and **MAE** measure average forecast error (RMSE penalises large misses more heavily by squaring them first); **MAPE** expresses that error as a percentage; **R²** measures the proportion of variation in realised volatility the model explains relative to simply forecasting the average every day (and, measured out-of-sample, can genuinely go negative if a model performs worse than that trivial benchmark); **Adjusted R²** further penalises a model for using more parameters; and **Directional Accuracy** measures how often a model correctly calls whether volatility will rise or fall from one day to the next.

**Reading the output.** EGARCH posts the lowest RMSE of the four models (0.0096), narrowly ahead of the Hybrid model (0.0098), with GJR-GARCH (0.0100) and plain GARCH (0.0109) behind. R² is negative for every single model — expected, and explained in the Discussion below, given how noisy a single day's squared return is as a proxy for latent variance. The Hybrid model stands out clearly on one metric in particular: **directional accuracy of 53.9%**, the only one of the four models to clear a coin flip, against roughly 48–49% for every GARCH-family model — a first hint that the LSTM has learned something genuinely different from, rather than merely a smoothed average of, the three GARCH inputs it was given.

### 6.2 VaR backtesting — where the real story is

The **Kupiec (1995) test** checks whether a model's overall VaR violation rate is statistically close to its nominal target (5% at the 95% confidence level, 1% at 99%); the **Christoffersen (1998) test** checks something the Kupiec test cannot — whether those violations are spread independently through time, or whether a violation on one day makes a violation on the very next day more likely, i.e. whether breaches **cluster**. A model can pass one test while failing the other, and this dissertation's central empirical finding turns on exactly that distinction.

**Kupiec results.** All four models produce broadly defensible violation rates: GARCH 3.11%/0.75%, EGARCH 3.98%/1.24%, GJR-GARCH 3.98%/1.12%, and Hybrid 5.35%/0.75% (95%/99% respectively, against 5.0%/1.0% targets). The Hybrid model's 95% violation rate is, in fact, the **closest to its nominal target of any of the four models** — the three GARCH-family models all sit on the conservative side (fewer violations than a perfectly calibrated model would produce), while the Hybrid model's rate sits almost exactly where theory says it should.

**Christoffersen results — the decisive finding.** Here the four models separate sharply. **Every single classical GARCH-family model fails the Christoffersen independence test at the 95% confidence level** (GARCH p = 0.043, EGARCH p = 0.040, GJR-GARCH p = 0.040 — all below the conventional 0.05 threshold), and GARCH and EGARCH fail it again at 99% (p = 0.032 and p = 0.004). In plain terms: their VaR breaches are not scattered randomly through the 804-day test period — they arrive in clusters, meaning a breach today makes another breach tomorrow more likely than pure chance would suggest, exactly the pattern a risk manager most wants to avoid, since it means a model's warnings go quiet precisely when a crisis is building and then arrive in an unhelpful cluster once it has already begun. **The Hybrid ML-GARCH model is the only one of the four that passes the Christoffersen test at both confidence levels**, and by a wide margin (p = 0.830 at 95%, p = 0.764 at 99%) — its breaches behave, statistically, like independent, unpredictable events, exactly what a well-specified risk model's breaches should look like.

**Expected Shortfall ratios.** The **ES ratio** compares the average *actual* loss on days a violation occurred against the model's own forecast of how severe such a loss should be — a ratio near 1 means a model's sense of "how bad it gets when it goes wrong" matches reality. The Hybrid model's ES ratios (0.945 at 95%, 1.031 at 99%) sit closer to 1 than any GARCH-family model's (clustered around 1.10–1.15, meaning real breaches tend to run moderately more severe than these classical models expect even when they correctly flag a breach is occurring).

Put together: **on the metric that matters most for whether a VaR model can actually be trusted in practice — not just how often it is wrong, but whether its mistakes are independent, unpredictable events rather than a clustering pattern a model itself should have anticipated — the Hybrid ML-GARCH model is unambiguously the strongest of the four**, despite EGARCH alone posting a marginally lower raw forecast error in Section 6.1. This is precisely the kind of result a single accuracy metric would have missed entirely.

### 6.3 Visualisations

The **GARCH-family comparison** chart (Section 4.3) shows the three classical forecasts moving closely together, with EGARCH visibly the smoothest of the three — a direct, visual counterpart to its highest persistence parameter ($\beta = 0.9695$). The **Hybrid-vs-GARCH overlay** (Section 6.3) adds the LSTM-refined forecast on top, drawn with a heavier line for emphasis, making it possible to see by eye where the Hybrid model's path diverges from a simple average of its three GARCH inputs — those divergences are exactly where the LSTM's refinement is doing genuine work rather than just echoing what it was given. The **VaR breach plots**, one each for the 95% and 99% confidence levels, plot the actual portfolio return against the GARCH and Hybrid VaR lines directly, marking every violation with a dot — the single most intuitive way to see the clustering finding from Section 6.2 directly: looking closely at the GARCH breach markers shows them bunching together in short windows, while the Hybrid model's breach markers are visibly more spread out across the test period. Finally, the **residual diagnostics** panel checks the Hybrid model's own errors for any remaining systematic pattern — an actual-vs-predicted scatter, a residual distribution, and a Q-Q plot, the same three diagnostic techniques introduced for the raw return series in the EDA notebook, now applied to a model's *errors* rather than to raw returns, to check whether those errors are reasonably well-behaved or still carry their own fat-tailed structure.

## 7. Model Persistence and Results Export

Every model is saved to the `models/` folder in a format matched to what it actually is. The three GARCH-family models are refit one final time on the entire available sample (in-sample and out-of-sample combined) and saved with Python's general-purpose `pickle` module, as the `arch` package's own documentation recommends for its result objects — `garch_model.pkl`, `egarch_model.pkl`, `gjr_garch_model.pkl`. The LSTM is saved exactly as it was trained and evaluated, without any further refitting (refitting it on the test period it was just backtested against would silently invalidate every result in Section 6), using **Keras's native `.keras` format** — `lstm_hybrid_model.keras` — which bundles the network's full architecture, its trained weights and its optimiser state into a single file, independent of which computational backend (PyTorch, in this project's case) happened to be used to train it. Its feature scaler is saved separately (`lstm_feature_scaler.joblib`), since a fresh forecast would need to be scaled with the exact same statistics before being handed to the network. Finally, `hybrid_var_results.pkl` bundles the feature list, lookback window, Duan smearing factor, Student-t degrees of freedom, and the full performance and backtest tables into one self-contained results object — everything another researcher would need to reproduce this notebook's headline findings without re-running the whole pipeline from scratch.

## 8. Discussion — what this dissertation actually found

The Discussion section of the technical notebook draws one central conclusion, and it is worth restating plainly here because it is easy to state imprecisely: **the Hybrid ML-GARCH model does not win this comparison by being the most accurate point forecaster** — EGARCH alone achieves a marginally lower RMSE. **It wins by being the only model whose VaR violations behave the way a well-specified risk model's violations should: independently through time, rather than clustering together.** Every classical GARCH-family model fails the Christoffersen independence test at the 95% confidence level; the Hybrid model passes it comfortably at both confidence levels tested. This is a genuinely meaningful result for the dissertation's thesis, and it is meaningful precisely *because* it would have been invisible to a simpler evaluation that stopped at comparing RMSE or even at comparing violation rates alone — it only shows up once a model's mistakes, not just their average frequency, are examined.

Why might the LSTM specifically have picked up on this pattern? It was given, as explicit inputs, the standardised residual from every one of the three GARCH variants at every point in its 21-day input window — a direct, numerical trace of exactly how surprising, and how recently, each model had been caught off guard. A network built to find patterns across a sequence has exactly the right shape to notice when a *cluster* of recent surprises is forming, something none of the three GARCH equations, each built around a single one-step recursion, can represent structurally. This is a plausible, mechanistic explanation for the result, consistent with the architecture's own design rather than a coincidence.

The limitations the technical notebook flags honestly are worth restating with the same care. The prediction target, next-day squared return, remains a noisy proxy for latent variance given the absence of intraday price data in this dissertation's scope — the most likely explanation for why R² is negative across all four models, a limitation shared equally by the classical and hybrid layers rather than a weakness specific to either. Training the LSTM once, rather than walk-forward on the same quarterly schedule as the GARCH layer beneath it, is a deliberate, literature-consistent trade-off between computational cost and statistical purity, not an oversight, and a fully walk-forward-retrained LSTM is a natural, well-motivated direction for further work — alongside extending the architecture to a multivariate GARCH specification that models time-varying correlation across the eight constituent equities directly, rather than through a single, fixed equally-weighted portfolio.